# RealFace：Colab側の生成受付サーバー
PCのGradio画面から写真を受け取り、ColabのGPUでFADINGを実行して画像を返します。
このノートブックを上から順に実行し、最後に表示される公開URLをPC側へ入力してください。

## 事前準備
Googleドライブに次のフォルダを置いてください。

| 配置先 | 内容 |
| --- | --- |
| マイドライブ/RealFace/FADING_stable | PCで使っているFADING_stableの本体とサブフォルダ |
| マイドライブ/RealFace/model/finetune_double_prompt_150_random | 学習済みモデル全体 |

Windowsの仮想環境は持ち込まず、Colab内で作ります。ランタイムはGPU（まずはT4）を選択してください。
PC側の画面や入力写真を事前にColabへコピーする必要はありません。写真は生成時に送信されます。
生成前の画像と生成後の画像はGoogleドライブのRealFace/resultsにも保存されます。

この構成も外部Web画面から生成処理を利用するため、無料Colabの制限を回避するものではありません。
有料プランとコンピューティングユニット残高がある環境を想定しています。
Colabは常設サーバーではなく、接続終了後は利用できません。
GPUと実際のモデルによる通し実行は未確認です。

基準：Shun0256/R03_RealFace、コミット1c7ed0a69ae31357b4c00b90ccd5d9862d6343a1。
PC側のapp.py・Viewは変更しません。


## 1. GPUが割り当てられているか確認


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import shutil
if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPUランタイムを選択してから、もう一度実行してください。")
check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(check.stdout or check.stderr)
if check.returncode != 0:
    raise RuntimeError("GPUランタイムを選択してから、もう一度実行してください。")


## 2. Googleドライブを接続し、ファイルを確認
接続の許可画面が出たら、ご自身のGoogleドライブを選んでください。


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/RealFace")
FADING_SOURCE = DRIVE_ROOT / "FADING_stable"
MODEL_SOURCE = DRIVE_ROOT / "model" / "finetune_double_prompt_150_random"

for path in (FADING_SOURCE / "age_editing.py", MODEL_SOURCE / "model_index.json"):
    if not path.is_file():
        raise FileNotFoundError(f"見つかりません：{path}\n事前準備のフォルダ配置を確認してください。")
for name in ("unet", "vae", "text_encoder", "tokenizer", "scheduler"):
    if not (MODEL_SOURCE / name).is_dir():
        raise FileNotFoundError(f"モデルのサブフォルダが不足しています：{name}")
print("FADING本体とモデルを確認しました。")


## 3. 実行用ファイルをColabへコピー
モデルは読み込みを速くするためColab内へコピーします。初回は時間がかかります。
元のドライブ上のFADING本体・モデルは書き換えません。
生成画像は `マイドライブ/RealFace/results` に保存します。
モデルやFADING本体を変更したときは、ランタイムを作り直してから実行してください。


In [ ]:
import shutil

BASE = Path("/content/realface_runtime")
PROJECT = BASE / "R03_RealFace"
FADING_DIR = BASE / "FADING_stable"
MODEL_DIR = BASE / "model" / "finetune_double_prompt_150_random"
BASE.mkdir(parents=True, exist_ok=True)

# 完了マーカーのないコピーは再開せず、上書きしてコピーし直す。
def copy_to_runtime(src, dst, marker_name):
    marker = BASE / marker_name
    if marker.exists() and dst.is_dir():
        print(f"コピー済み：{dst}")
        return
    print(f"コピー中：{src}", flush=True)
    shutil.copytree(src, dst, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns(".git", "__pycache__", ".venv*"))
    marker.write_text("complete", encoding="utf-8")

copy_to_runtime(FADING_SOURCE, FADING_DIR, "fading_copy_complete")
copy_to_runtime(MODEL_SOURCE, MODEL_DIR, "model_copy_complete")


In [ ]:
#@title 現在のプロジェクトのControllerと受付APIを配置
PROJECT_SOURCES = {'Controll/exe_fading.py': 'from pathlib import Path\nfrom uuid import uuid4\nimport math\nimport subprocess\nimport os\nimport re\nfrom queue import Queue, Empty\nfrom threading import Thread\n\nfrom PIL import Image, ImageOps\n\n\n# ===== フォルダ・ファイルの場所 =====\n\n# R03_RealFace\nPROJECT_DIR = Path(__file__).resolve().parent.parent\n\n# R4S115（R03_RealFaceの1つ上）\nROOT_DIR = PROJECT_DIR.parent\n\n# FADING本体\nFADING_DIR = ROOT_DIR / "FADING_stable"\n\n# FADING用の仮想環境のPython\nPYTHON_PATH = ROOT_DIR / ".venv_fading" / "Scripts" / "python.exe"\n\n# 学習済みモデル\nMODEL_DIR = ROOT_DIR / "model" / "finetune_double_prompt_150_random"\n\n# 入力画像・生成画像の保存先\nWORK_DIR = PROJECT_DIR / "fading_results"\n\n\n# ===== 入力内容の確認 =====\n\ndef check_input(image_path, current_age, target_age):\n    """画像と年齢を確認し、処理に使える形式で返す。"""\n\n    if image_path is None or str(image_path).strip() == "":\n        raise ValueError("顔写真を選択してください。")\n\n    image_path = Path(image_path).resolve()\n\n    if not image_path.is_file():\n        raise ValueError("選択した画像ファイルが見つかりません。")\n\n    # 画像として読み込めるか確認\n    try:\n        with Image.open(image_path) as image:\n            image.verify()\n    except (OSError, ValueError, SyntaxError) as error:\n        raise ValueError(\n            "画像を読み込めません。別の画像を選択してください。"\n        ) from error\n\n    ages = []\n\n    for label, value in [\n        ("現在の年齢", current_age),\n        ("変換後の年齢", target_age),\n    ]:\n        try:\n            number = float(value)\n        except (TypeError, ValueError, OverflowError) as error:\n            raise ValueError(\n                f"{label}を数値で入力してください。"\n            ) from error\n\n        if not math.isfinite(number) or not number.is_integer():\n            raise ValueError(f"{label}は整数で入力してください。")\n\n        # このアプリでは0～100歳に制限\n        if not 0 <= number <= 100:\n            raise ValueError(\n                f"{label}は0～100歳で入力してください。"\n            )\n\n        ages.append(int(number))\n\n    current_age, target_age = ages\n\n    if current_age == target_age:\n        raise ValueError("現在と異なる年齢を指定してください。")\n\n    return image_path, current_age, target_age\n\n\n# ===== 入力画像の準備 =====\n\ndef prepare_image(image_path):\n    """画像の向き・色形式を整え、実行専用フォルダに保存する。"""\n\n    # 実行ごとに別のフォルダを作る\n    run_dir = WORK_DIR / uuid4().hex\n    run_dir.mkdir(parents=True, exist_ok=False)\n\n    # 入力画像の保存先\n    prepared_path = run_dir / "input.png"\n\n    # 生成画像の保存先\n    output_dir = run_dir / "output"\n    output_dir.mkdir()\n\n    with Image.open(image_path) as image:\n        # スマホ写真などの向き情報を反映\n        image = ImageOps.exif_transpose(image)\n\n        # RGB形式に統一\n        image = image.convert("RGB")\n\n        # 元画像を上書きせず、別の画像として保存\n        image.save(prepared_path, format="PNG")\n\n    return prepared_path, output_dir\n\n\n# ===== FADINGの実行 =====\n\ndef run_fading_stream(\n    image_path,\n    current_age,\n    target_age,\n    output_dir,\n    gender,\n):\n    """FADINGを実行し、処理が終わるまで待つ。"""\n\n    if gender not in ("male", "female"):\n        raise ValueError(\n            "genderはmaleまたはfemaleを指定してください。"\n        )\n\n    script_path = FADING_DIR / "age_editing.py"\n\n    if not PYTHON_PATH.is_file():\n        raise FileNotFoundError(\n            f"仮想環境のPythonが見つかりません：{PYTHON_PATH}"\n        )\n\n    if not script_path.is_file():\n        raise FileNotFoundError(\n            f"FADINGの実行ファイルが見つかりません：{script_path}"\n        )\n\n    if not (MODEL_DIR / "model_index.json").is_file():\n        raise FileNotFoundError(\n            "学習済みモデルのmodel_index.jsonが見つかりません。"\n            f"MODEL_DIRを確認してください：{MODEL_DIR}"\n        )\n\n    # FADINGへ渡す実行コマンド\n    command = [\n        str(PYTHON_PATH),\n        "-u",\n        str(script_path),\n        "--image_path", image_path.as_posix(),\n        "--age_init", str(current_age),\n        "--gender", gender,\n        "--save_aged_dir", output_dir.as_posix(),\n        "--specialized_path", MODEL_DIR.as_posix(),\n        "--target_ages", str(target_age),\n    ]\n\n    # tqdmの進捗はstderrに出るため、stdoutとまとめて読み取る。\n    env = os.environ.copy()\n    env["PYTHONIOENCODING"] = "utf-8"\n    process = subprocess.Popen(\n        command, cwd=str(FADING_DIR), stdout=subprocess.PIPE,\n        stderr=subprocess.STDOUT, text=True, encoding="utf-8",\n        errors="replace", bufsize=1, env=env,\n    )\n    messages = Queue()\n\n    def read_output():\n        try:\n            # text=Trueで、tqdmの\\rによる更新も行単位で読める。\n            for line in process.stdout:\n                messages.put(line)\n        finally:\n            messages.put(None)\n\n    reader = Thread(target=read_output, daemon=True)\n    reader.start()\n    try:\n        while True:\n            try:\n                line = messages.get(timeout=1)\n            except Empty:\n                # ログのない計算中も、Web側の経過時間を更新する。\n                yield {"kind": "heartbeat"}\n                continue\n            if line is None:\n                break\n            line = re.sub(r"\\x1b\\[[0-?]*[ -/]*[@-~]", "", line).strip()\n            if line:\n                print(line, flush=True)\n                yield {"kind": "log", "text": line}\n        code = process.wait()\n        if code != 0:\n            raise RuntimeError(\n                f"FADINGの処理に失敗しました（終了コード：{code}）。"\n                "実行ログまたはターミナルを確認してください。"\n            )\n    finally:\n        # ジェネレーターが閉じられたときにも子プロセスを残さない。\n        if process.poll() is None:\n            process.terminate()\n            try:\n                process.wait(timeout=5)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait()\n        reader.join(timeout=2)\n        process.stdout.close()\n\n\ndef run_fading(image_path, current_age, target_age, output_dir, gender):\n    """従来と同じ呼び方も維持する。"""\n    for _ in run_fading_stream(image_path, current_age, target_age, output_dir, gender):\n        pass\n\n\n# ===== 生成画像の確認 =====\n\ndef find_result_image(image_path, target_age, output_dir):\n    """生成画像が存在することを確認し、パスを文字列で返す。"""\n\n    # 例：入力がinput.png、変換後が40歳ならinput_40.png\n    result_path = output_dir / f"{image_path.stem}_{target_age}.png"\n\n    if not result_path.is_file():\n        raise FileNotFoundError(\n            "処理は終了しましたが、生成画像が見つかりません："\n            f"{result_path}"\n        )\n\n    return str(result_path)\n\n\n# ===== 全体の処理 =====\n\ndef execute_fading(\n    image_path,\n    current_age,\n    target_age,\n    gender="male",\n):\n    """入力確認・画像準備・FADING実行・結果取得を順番に行う。"""\n    if os.environ.get("REALFACE_BACKEND") == "colab":\n        from Controll.colab_client import execute_fading as execute_remote\n        return execute_remote(image_path, current_age, target_age, gender)\n\n    image_path, current_age, target_age = check_input(\n        image_path,\n        current_age,\n        target_age,\n    )\n\n    prepared_path, output_dir = prepare_image(image_path)\n\n    run_fading(\n        prepared_path,\n        current_age,\n        target_age,\n        output_dir,\n        gender,\n    )\n\n    result_path = find_result_image(\n        prepared_path,\n        target_age,\n        output_dir,\n    )\n\n    return result_path\n\n\ndef execute_fading_stream(image_path, current_age, target_age, gender="male"):\n    """進捗イベントをyieldし、最後に結果画像のパスを通知する。"""\n    if os.environ.get("REALFACE_BACKEND") == "colab":\n        from Controll.colab_client import execute_fading_stream as execute_remote_stream\n        yield from execute_remote_stream(image_path, current_age, target_age, gender)\n        return\n    yield {"kind": "stage", "text": "入力内容を確認しています"}\n    image_path, current_age, target_age = check_input(\n        image_path, current_age, target_age,\n    )\n    yield {"kind": "stage", "text": "入力画像を準備しています"}\n    prepared_path, output_dir = prepare_image(image_path)\n    yield {"kind": "stage", "text": "FADINGを起動しています"}\n    yield from run_fading_stream(\n        prepared_path, current_age, target_age, output_dir, gender,\n    )\n    yield {"kind": "stage", "text": "生成画像を確認しています"}\n    result_path = find_result_image(prepared_path, target_age, output_dir)\n    yield {"kind": "result", "path": result_path, "original_path": str(prepared_path)}\n', 'remote_server.py': '"""Colab側の画像生成API。画面はPC側、FADINGだけここで実行。"""\nimport re\nfrom contextlib import closing\nimport gradio as gr\nfrom Controll.exe_fading import execute_fading_stream\n\n\ndef generate(image_path, current_age, target_age, gender, request_id):\n    if not re.fullmatch(r"[0-9a-f]{32}", request_id):\n        raise gr.Error("依頼IDの形式が不正です。")\n    try:\n        # API利用でも入力チェックはController側で再実行する。\n        with closing(execute_fading_stream(image_path, current_age, target_age, gender)) as events:\n            for event in events:\n                file_path = event["path"] if event["kind"] == "result" else None\n                # サーバーの絶対パスは、PC向けの進捗データには含めない。\n                public_event = {k: v for k, v in event.items() if k not in ("path", "original_path")}\n                yield {"request_id": request_id, "event": public_event}, file_path\n    except Exception as error:\n        yield {"request_id": request_id,\n               "event": {"kind": "error", "text": str(error)}}, None\n\n\ndef build_server():\n    with gr.Blocks(title="RealFace 生成受付") as server:\n        gr.Markdown("# RealFace 生成受付\\n操作はPC側のRealFace画面で行ってください。")\n        # これらはAPIの入出力定義。ユーザー向け画面はPC上にある。\n        photo = gr.File(type="filepath", visible=False)\n        current = gr.Number(visible=False)\n        target = gr.Number(visible=False)\n        gender = gr.Textbox(visible=False)\n        request_id = gr.Textbox(visible=False)\n        metadata = gr.JSON(visible=False)\n        result = gr.File(visible=False)\n        trigger = gr.Button(visible=False)\n        trigger.click(generate, [photo, current, target, gender, request_id],\n                      [metadata, result], api_name="generate",\n                      concurrency_limit=1, concurrency_id="fading")\n    return server.queue(max_size=8)\n'}
for relative, content in PROJECT_SOURCES.items():
    path = PROJECT / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
print("ControllerとAPIを配置しました。")


## 4. FADING用のPython環境を作成
FADING_stableの公開requirementsを基に、Python 3.10・PyTorch 2.0.1・diffusers 0.27.1を使用します。
古いdiffusersに必要なAPIを維持するためhuggingface_hubを0.25.2に固定し、NumPyも1系に固定しています。
Gradioとは別環境なので、GradioのインストールでFADINGの依存関係を変更しません。
手元のFADING本体に独自の変更や追加ライブラリがある場合は、追加調整が必要になる場合があります。


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
VENV = BASE / ".venv_fading"
FADING_PYTHON = VENV / "bin" / "python"
if not FADING_PYTHON.is_file():
    subprocess.run(UV + ["python", "install", "3.10"], check=True)
    subprocess.run(UV + ["venv", "--python", "3.10", str(VENV)], check=True)

subprocess.run(UV + ["pip", "install", "--python", str(FADING_PYTHON),
    "torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2",
    "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
subprocess.run(UV + ["pip", "install", "--python", str(FADING_PYTHON),
    "accelerate==0.28.0", "diffusers==0.27.1", "transformers==4.36.2",
    "huggingface_hub==0.25.2", "numpy==1.26.4", "opencv-python==4.8.1.78",
    "Pillow==10.4.0", "matplotlib==3.8.4", "scipy==1.11.4", "ipython==8.29.0"], check=True)
subprocess.run(UV + ["pip", "check", "--python", str(FADING_PYTHON)], check=True)
print("FADING用の環境を作成しました。")


## 5. GPU・FADINGの読み込み確認とパス設定
`GPU確認OK` とコマンドのヘルプが表示されたら準備完了です。モデルを使う実際の生成は次のセルで行います。


In [ ]:
probe = """
import torch, diffusers, transformers, huggingface_hub
print('PyTorch:', torch.__version__)
print('diffusers:', diffusers.__version__)
print('transformers:', transformers.__version__)
print('huggingface_hub:', huggingface_hub.__version__)
assert torch.cuda.is_available(), 'FADING用環境からGPUを認識できません。'
x = torch.ones(2, device='cuda')
print('GPU確認OK:', torch.cuda.get_device_name(0), x.sum().item())
"""
subprocess.run([str(FADING_PYTHON), "-c", probe], check=True)
subprocess.run([str(FADING_PYTHON), str(FADING_DIR / "age_editing.py"), "--help"],
               cwd=str(FADING_DIR), check=True)

os.environ.update({
    "REALFACE_FADING_DIR": str(FADING_DIR),
    "REALFACE_MODEL_DIR": str(MODEL_DIR),
    "REALFACE_PYTHON": str(FADING_PYTHON),
    "REALFACE_WORK_DIR": str(DRIVE_ROOT / "results"),
})
# 設定変更後にも、Controllerのパスを読み直せるようにする。
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import importlib
import Controll.exe_fading as fading_controller
importlib.reload(fading_controller)
# Windows向けの既存コードを維持し、Colab側の起動時だけパスを設定する。
os.environ["REALFACE_BACKEND"] = "local"
fading_controller.FADING_DIR = FADING_DIR
fading_controller.MODEL_DIR = MODEL_DIR
fading_controller.PYTHON_PATH = FADING_PYTHON
fading_controller.WORK_DIR = DRIVE_ROOT / "results"
assert fading_controller.PYTHON_PATH.is_file()
print("準備完了。次の受付サーバー起動へ進んでください。")


## 6. 受付サーバーを起動
パスワードを決めて入力してください。ユーザー名は`realface`です。
表示された `https://....gradio.live` の公開URLをコピーし、PC側の`start_colab.py`へ入力します。
このセルはサーバーを稼働させるため実行中のままになります。
PC側の生成ボタンから依頼すると、ここにFADINGのログが流れます。


In [ ]:
import inspect
from getpass import getpass
subprocess.run([sys.executable, "-m", "pip", "install", "gradio==6.31.0"], check=True)
import gradio as gr
if gr.__version__ != "6.31.0":
    raise RuntimeError("Gradioの別バージョンが読み込み済みです。ランタイムを再起動してください。")
from remote_server import build_server
password = getpass("PCから接続するためのパスワード：")
if not password:
    raise ValueError("パスワードを入力してください。")
if "api_server" in globals():
    api_server.close()
api_server = build_server()
print("接続ユーザー名：realface。以下の公開URLをPC側へ入力してください。")
api_server.launch(share=True, auth=("realface", password),
                  inbrowser=False, inline=False, debug=True,
                  allowed_paths=[str(DRIVE_ROOT / "results")])


## 停止・再起動
終了するときは「ランタイム → 接続を解除してランタイムを削除」で処理を終了します。
再接続後は準備セルを再実行します。公開URLが変わった場合は、PC側も起動し直して新しいURLを入力します。
PCのブラウザを閉じるだけでColab側の生成が確実に停止するとは限りません。
通信切断時は自動再送しません。Colabのログを確認してから再度生成してください。

## 検証範囲
PCとColabの役割を分けた2プロセス相当のローカルHTTP接続で、認証・画像送信・進捗・画像取得・エラーを検証。
本物のColab共有URL、GPU、モデルを使った通し実行は未確認。

参考：[Colab FAQ](https://research.google.com/colaboratory/faq.html) /
[Gradio Python Client](https://gradio.app/guides/getting-started-with-the-python-client)
